# SP01 — Visium Analysis In Depth

**Tools:** `scanpy`, `squidpy`  
**Dataset:** Mouse brain Visium H&E (~2,695 spots, `sq.datasets.visium_hne_adata()`)  
**Papers:** [Stahl et al. 2016, Science](https://doi.org/10.1126/science.aaf2403) · [Palla et al. 2022, Nature Methods](https://doi.org/10.1038/s41592-021-01358-2)

**Prerequisites:** `theis_ecosystem/T02` for squidpy basics (spatial neighbors, Moran's I, nhood enrichment). This notebook goes deeper: spatial-specific QC, differential expression between tissue regions, and image-based feature extraction.

---

## What Visium analysis adds beyond standard scRNA-seq

Visium spots are **not cells** — each 55 µm spot contains 5–30 cells from the tissue section. This changes several aspects of analysis:

| scRNA-seq | Visium |
|-----------|--------|
| QC: nGenes, %MT per cell | QC: also spatial distribution of metrics (artifacts are spatially clustered) |
| Clustering: cell type identity | Clustering: tissue regions / layers |
| DE: cell type A vs. B | DE: region A vs. B (anatomically defined) |
| SVGs: n/a | SVGs: which genes vary spatially across tissue? |
| LR: cell-cell (estimated) | LR: cell-cell (physically constrained by distance) |

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
import matplotlib.pyplot as plt
import seaborn as sns

sc.settings.set_figure_params(dpi=80, facecolor='white')
print(f'scanpy {sc.__version__}  |  squidpy {sq.__version__}')

## 1. Load and Spatial QC

In [ ]:
adata = sq.datasets.visium_hne_adata()
print(adata)

# Mark mitochondrial genes (mouse: lowercase 'mt-')
adata.var['mt'] = adata.var_names.str.startswith('mt-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)

In [ ]:
# Spatial QC: plot standard metrics on tissue
# Artifacts often cluster spatially — low-quality regions of tissue,
# necrotic areas, edges. This is invisible in a violin plot.
sq.pl.spatial_scatter(
    adata,
    color=['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    ncols=3, size=1.5, figsize=(13, 4),
    title=['Genes per spot', 'Total UMI', '% Mitochondrial'],
)

In [ ]:
# Identify and remove low-quality spots
# Spatial artifacts: very low total counts (tissue damage, edge spots)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].hist(adata.obs['total_counts'], bins=50, edgecolor='k', linewidth=0.5)
axes[0].axvline(500, color='red', linestyle='--', label='threshold=500')
axes[0].set_xlabel('Total counts')
axes[0].set_title('UMI distribution')
axes[0].legend()

axes[1].hist(adata.obs['n_genes_by_counts'], bins=50, edgecolor='k', linewidth=0.5)
axes[1].axvline(250, color='red', linestyle='--', label='threshold=250')
axes[1].set_xlabel('Genes detected')
axes[1].set_title('Gene detection')
axes[1].legend()
plt.tight_layout()
plt.show()

# Filter
n_before = adata.n_obs
adata = adata[adata.obs['total_counts'] > 500]
adata = adata[adata.obs['n_genes_by_counts'] > 250]
print(f'Removed {n_before - adata.n_obs} low-quality spots')

In [ ]:
# Standard normalization
sc.pp.filter_genes(adata, min_cells=10)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=3000)
sc.pp.pca(adata)
sc.pp.neighbors(adata, n_pcs=30)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.4, key_added='leiden')
print('Leiden clusters:', adata.obs['leiden'].nunique())

In [ ]:
# Compare UMAP (expression-only) with spatial tissue view side-by-side
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sc.pl.umap(adata, color='leiden', legend_loc='on data', ax=axes[0], show=False)
sq.pl.spatial_scatter(adata, color='leiden', size=1.5, ax=axes[1], figsize=(5, 4))
axes[1].set_title('Leiden clusters on tissue')
plt.tight_layout()
plt.show()

## 2. Spatially Variable Genes (SVGs) — Going Deeper

`theis_ecosystem/T02` showed Moran's I for ranking SVGs. Here we look at the biology of the top SVGs and discuss alternative methods.

In [ ]:
# Rebuild spatial neighbors (needed for autocorrelation)
sq.gr.spatial_neighbors(adata, coord_type='visium', n_neighs=6)

# Moran's I on highly variable genes
sq.gr.spatial_autocorr(
    adata,
    mode='moran',
    genes=adata.var_names[adata.var['highly_variable']],
    n_perms=100,
    n_jobs=1,
)
moranI = adata.uns['moranI'].sort_values('I', ascending=False)

# Categorize SVGs by Moran's I magnitude
moranI['category'] = pd.cut(
    moranI['I'],
    bins=[-1, 0.1, 0.3, 0.5, 1.0],
    labels=['low', 'moderate', 'high', 'very_high'],
)
print('SVG categories:')
print(moranI['category'].value_counts())

In [ ]:
# Top SVGs: visualize on tissue
top_svgs = moranI.head(6).index.tolist()
sq.pl.spatial_scatter(
    adata, color=top_svgs, ncols=3, size=1.5,
    color_map='magma', figsize=(12, 8),
    title=[f'{g}  I={moranI.loc[g, "I"]:.2f}' for g in top_svgs],
)

In [ ]:
# Geary's C: alternative to Moran's I
# Geary's C ≈ 0: high spatial autocorrelation (similar to Moran's I ≈ 1)
# Useful for detecting local spatial heterogeneity (Moran's detects global)
sq.gr.spatial_autocorr(
    adata,
    mode='geary',
    genes=adata.var_names[adata.var['highly_variable']],
    n_perms=100,
    n_jobs=1,
)
gearyC = adata.uns['gearyC'].sort_values('C')
print('Lowest Geary\'s C (most spatially variable):')
print(gearyC.head(10)[['C', 'pval_norm']])

## 3. Spatial Differential Expression Between Tissue Regions

Unlike scRNA-seq where you compare *cell types*, Visium DE typically compares *anatomically defined regions*. Approach: use the cluster annotations as region labels, then run pseudobulk DE between regions.

In [ ]:
# Map Leiden clusters to anatomical regions (using the pre-annotated 'cluster' column)
# In a real experiment you would annotate manually or use Allen Brain Atlas registration
print('Annotated regions in this dataset:')
print(adata.obs['cluster'].value_counts())

In [ ]:
# Per-region marker genes: which genes define each anatomical region?
sc.tl.rank_genes_groups(
    adata,
    groupby='cluster',
    method='wilcoxon',
    use_raw=False,
)

# Top markers for each region
sc.pl.rank_genes_groups_dotplot(
    adata,
    groupby='cluster',
    n_genes=4,
    standard_scale='var',
    figsize=(12, 5),
)

In [ ]:
# Spatial DE: Hippocampus vs. Cortex — a targeted regional comparison
# Select spots from each region
hippo = adata[adata.obs['cluster'] == 'Hippocampus'].copy()
cortex = adata[adata.obs['cluster'].str.contains('Cortex')].copy()

hippo.obs['region'] = 'Hippocampus'
cortex.obs['region'] = 'Cortex'

import anndata as ad
combined = ad.concat([hippo, cortex])
sc.tl.rank_genes_groups(combined, groupby='region', method='wilcoxon')

# Top DE genes
result = sc.get.rank_genes_groups_df(combined, group='Hippocampus').head(20)
print('Top Hippocampus vs. Cortex markers:')
print(result[['names', 'logfoldchanges', 'pvals_adj']].head(10))

In [ ]:
# Visualize top spatial DE genes on tissue
top_regional = result['names'].head(4).tolist()
sq.pl.spatial_scatter(
    adata, color=top_regional, ncols=4, size=1.5,
    color_map='magma', figsize=(14, 3.5),
    title=[f'{g}\n(Hippo vs Cortex)' for g in top_regional],
)

## 4. H&E Image Feature Extraction

The H&E image contains morphological information beyond gene expression: cell density, tissue texture, staining intensity. Squidpy can extract these features per spot for integration with transcriptomics.

In [ ]:
# Image container: wraps the H&E image with spot cropping utilities
from squidpy.im import ImageContainer

# Build ImageContainer from the AnnData H&E image
library_id = list(adata.uns['spatial'].keys())[0]
img_arr = adata.uns['spatial'][library_id]['images']['hires']
scale = adata.uns['spatial'][library_id]['scalefactors']['tissue_hires_scalef']

img = ImageContainer(img_arr, library_id=library_id, scale=scale)
print('ImageContainer:', img)

In [ ]:
# Extract morphological features per spot:
# summary: mean/std/percentiles of each channel in the spot neighborhood
# histogram: pixel intensity distribution per channel
# texture: GLCM (Gray-Level Co-occurrence Matrix) — captures tissue texture
sq.im.calculate_image_features(
    adata,
    img,
    features='summary',           # fast; use 'texture' for GLCM features
    key_added='img_features',
    n_jobs=1,
    show_progress_bar=True,
)
print('Image features added to adata.obsm["img_features"]:')
print(adata.obsm['img_features'].shape)
print(adata.obsm['img_features'].head())

In [ ]:
# Cluster using image features alone (morphology-based clusters)
# Compare to gene expression clusters
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

img_feats = adata.obsm['img_features'].values
img_feats_scaled = StandardScaler().fit_transform(img_feats)
img_pca = PCA(n_components=10).fit_transform(img_feats_scaled)
adata.obsm['X_img_pca'] = img_pca

sc.pp.neighbors(adata, use_rep='X_img_pca', key_added='img_neighbors')
sc.tl.leiden(adata, neighbors_key='img_neighbors', resolution=0.3, key_added='leiden_img')

# Side-by-side: expression vs morphology clustering
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sq.pl.spatial_scatter(adata, color='leiden', size=1.5, ax=axes[0], figsize=(5, 4))
axes[0].set_title('Expression-based clusters')
sq.pl.spatial_scatter(adata, color='leiden_img', size=1.5, ax=axes[1], figsize=(5, 4))
axes[1].set_title('Morphology-based clusters (H&E)')
plt.tight_layout()
plt.show()

In [ ]:
# Combined: neighbors graph from expression + image features
# Concatenate expression PCA and image PCA
import numpy as np
combined_feats = np.hstack([adata.obsm['X_pca'][:, :20], img_pca])
adata.obsm['X_combined'] = combined_feats

sc.pp.neighbors(adata, use_rep='X_combined', key_added='combined_neighbors')
sc.tl.leiden(adata, neighbors_key='combined_neighbors', resolution=0.4, key_added='leiden_combined')

sq.pl.spatial_scatter(adata, color='leiden_combined', size=1.5, figsize=(5, 5),
                       title='Combined expression + morphology clusters')

In [ ]:
# Save for downstream notebooks
adata.write_h5ad('../data/visium_processed.h5ad')
print('Saved to ../data/visium_processed.h5ad')

---
## Summary

| Analysis | Key function | Output |
|----------|-------------|--------|
| Spatial QC | `sq.pl.spatial_scatter(color=['pct_counts_mt'])` | Spatial artifact detection |
| SVGs | `sq.gr.spatial_autocorr(mode='moran')` | `.uns['moranI']` |
| SVGs (local) | `sq.gr.spatial_autocorr(mode='geary')` | `.uns['gearyC']` |
| Region DE | `sc.tl.rank_genes_groups(groupby='region')` | `.uns['rank_genes_groups']` |
| Image features | `sq.im.calculate_image_features(adata, img)` | `.obsm['img_features']` |

**Next:** SP02 — Spatial domain detection with BANKSY (spatially-aware clustering beyond expression).